# 竞争者订单数据查询（orders.db）

用于查看已同步到 `db/orders.db` 的竞争者数据表。

默认支持两类前缀：
- `comp_turtle_`（订单数据（龟））
- `comp_v42_`（订单数据（4-2））

> 如果暂时看不到数据，请先运行同步脚本：
> `analysis/.venv/bin/python db/scripts/sync_competitor_csv_to_orders.py --db-path db/orders.db --recursive`

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Hiragino Sans GB', 'Heiti SC', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

DB_PATH = '/Users/yy/.openclaw/workspace/db/orders.db'
conn = sqlite3.connect(DB_PATH)
print('Connected:', DB_PATH)

In [ ]:
# 参数区
TARGET_PREFIX = None    # 可选: 'comp_turtle_' / 'comp_v42_' / None(全部)
TARGET_TABLE = None     # 例如: 'comp_turtle_k1_订单数据_2026_03_29'，None=自动选第一个
PREVIEW_LIMIT = 200

print('TARGET_PREFIX =', TARGET_PREFIX)
print('TARGET_TABLE  =', TARGET_TABLE)
print('PREVIEW_LIMIT =', PREVIEW_LIMIT)

In [ ]:
# 1) 查看竞争者相关表 + 同步状态表
where_prefix = ''
params = []
if TARGET_PREFIX:
    where_prefix = 'AND name LIKE ?'
    params.append(TARGET_PREFIX + '%')

sql_tables = f"""
SELECT name AS table_name
FROM sqlite_master
WHERE type='table'
  AND (name LIKE 'comp_turtle_%' OR name LIKE 'comp_v42_%')
  {where_prefix}
ORDER BY name
"""
df_tables = pd.read_sql_query(sql_tables, conn, params=params)
display(df_tables)

sql_states = """
SELECT '__csv_sync_state_comp_turtle' AS state_table, * FROM __csv_sync_state_comp_turtle
UNION ALL
SELECT '__csv_sync_state_comp_turtle_xlsx' AS state_table, * FROM __csv_sync_state_comp_turtle_xlsx
UNION ALL
SELECT '__csv_sync_state_comp_v42' AS state_table, * FROM __csv_sync_state_comp_v42
UNION ALL
SELECT '__csv_sync_state_comp_v42_xlsx' AS state_table, * FROM __csv_sync_state_comp_v42_xlsx
"""
try:
    df_states = pd.read_sql_query(sql_states, conn)
    display(df_states)
except Exception as e:
    print('状态表暂不可用（可能尚未创建）:', e)

In [ ]:
# 2) 预览单表结构 + 前N行
if TARGET_TABLE:
    selected_table = TARGET_TABLE
elif 'df_tables' in globals() and not df_tables.empty:
    selected_table = df_tables['table_name'].iloc[0]
else:
    selected_table = None

print('selected_table =', selected_table)

if selected_table:
    df_schema = pd.read_sql_query(f"PRAGMA table_info('{selected_table}')", conn)
    display(df_schema)

    df_preview = pd.read_sql_query(
        f"SELECT * FROM '{selected_table}' LIMIT {int(PREVIEW_LIMIT)}",
        conn
    )
    display(df_preview)
else:
    print('暂无竞争者数据表。')

In [ ]:
# 3) 统计每个竞争者表的行数
if 'df_tables' in globals() and not df_tables.empty:
    rows = []
    for t in df_tables['table_name']:
        n = pd.read_sql_query(f"SELECT COUNT(1) AS n FROM '{t}'", conn)['n'].iloc[0]
        rows.append({'table_name': t, 'row_count': int(n)})
    df_counts = pd.DataFrame(rows).sort_values('row_count', ascending=False).reset_index(drop=True)
    display(df_counts)
else:
    df_counts = pd.DataFrame(columns=['table_name', 'row_count'])
    print('暂无可统计的竞争者表。')

In [ ]:
# 4) 可视化：各竞争者表行数
if 'df_counts' in globals() and not df_counts.empty:
    show_n = min(20, len(df_counts))
    d = df_counts.head(show_n).copy()

    plt.figure(figsize=(12, 4 + show_n * 0.15))
    sns.barplot(data=d, y='table_name', x='row_count', palette='Blues_r')
    plt.title(f'竞争者表行数 Top {show_n}')
    plt.xlabel('row_count')
    plt.ylabel('table_name')
    plt.tight_layout()
    plt.show()
else:
    print('请先同步竞争者数据，或先运行上一个统计单元。')

In [ ]:
# 可选：关闭连接
# conn.close()